# SETU.AI — Train your own model (Colab GPU)

Fine-tunes a small open model (Qwen2.5-3B-Instruct by default) on SETU.AI's
four AI tasks — training capsules, skill assessments, multilingual resumes,
matching — using QLoRA, then exports a quantized GGUF you download and load
into Ollama on your own machine.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (free tier is
enough; A100 if you have Colab Pro and want it faster).

**Optional, for a better model:** if you've been running SETU.AI with
`LOG_TRAINING_DATA=true`, upload your real `backend/data/training_log.jsonl`
in the cell marked below to blend real usage in with the seed data. Not
required for a first end-to-end run — the embedded seed data alone is enough
to prove the pipeline works.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q -U pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft accelerate bitsandbytes


## 3. Seed dataset (embedded — no upload needed)

This is the same 13-example seed set from `ai-engine/data/seed.jsonl` in your
project zip, covering all 4 tasks (capsules, assessments, resumes in
en/hi/ta, matching).

In [ ]:
SEED_JSONL_LINES = [
"{\"system\": \"You are a vocational trainer generating a micro-upskilling capsule for a new hire at an Indian MSME, based on the owner's description of the exact task, machine, or SOP. Return strict JSON only, no markdown. Schema: {\\\"role\\\": short job title, \\\"requiredTrade\\\": closest matching trade name, \\\"skillsNeeded\\\": array of 3-5 short tags, \\\"steps\\\": array of 4-6 objects {\\\"title\\\": short imperative title, \\\"instruction\\\": one to two sentences specific to what the owner described}, \\\"quizQuestions\\\": array of 2-3 objects {\\\"question\\\": string, \\\"answer\\\": string}}. Never write generic trade advice \u2014 every step must trace back to specific details the owner gave.\", \"user\": \"MSME name: Sri Balaji Welding Works\\nSector: Fabrication\\nLocation: Coimbatore, TN\\nWage band: \u20b912,000-15,000/mo\\nOwner's description of the exact task/machine/SOP: \\\"We need someone to operate our arc welding machine for MS angle frames used in gates and window grills. Must check current settings before starting and always wear the welding shield.\\\"\\n\\nGenerate the capsule as JSON.\", \"response\": {\"role\": \"Arc Welder Assistant\", \"requiredTrade\": \"Welder (Arc/MMA)\", \"skillsNeeded\": [\"Arc welding\", \"MS fabrication\", \"Electrode selection\", \"Safety compliance\"], \"steps\": [{\"title\": \"Check machine settings\", \"instruction\": \"Verify the arc welding machine's current setting matches the electrode gauge before every job.\"}, {\"title\": \"Wear the welding shield\", \"instruction\": \"Put on the welding shield and gloves before striking the arc, every single time, no exceptions.\"}, {\"title\": \"Set up the MS angle frame\", \"instruction\": \"Clamp the MS angle sections for the gate or grill in position and tack-weld the corners first.\"}, {\"title\": \"Run the full weld seam\", \"instruction\": \"Weld the full seam in short, steady passes, checking penetration on each angle joint.\"}, {\"title\": \"Inspect the finished frame\", \"instruction\": \"Check the gate or grill frame for straightness and clean off slag before handover.\"}], \"quizQuestions\": [{\"question\": \"What must you check before starting the arc welding machine?\", \"answer\": \"That the current setting matches the electrode gauge being used.\"}, {\"question\": \"What must you wear before striking the arc?\", \"answer\": \"The welding shield and gloves.\"}]}}",
"{\"system\": \"You are a vocational trainer generating a micro-upskilling capsule for a new hire at an Indian MSME, based on the owner's description of the exact task, machine, or SOP. Return strict JSON only, no markdown. Schema: {\\\"role\\\": short job title, \\\"requiredTrade\\\": closest matching trade name, \\\"skillsNeeded\\\": array of 3-5 short tags, \\\"steps\\\": array of 4-6 objects {\\\"title\\\": short imperative title, \\\"instruction\\\": one to two sentences specific to what the owner described}, \\\"quizQuestions\\\": array of 2-3 objects {\\\"question\\\": string, \\\"answer\\\": string}}. Never write generic trade advice \u2014 every step must trace back to specific details the owner gave.\", \"user\": \"MSME name: Anand Tailors\\nSector: Garments\\nLocation: Jaipur, RJ\\nWage band: \u20b99,000-11,000/mo\\nOwner's description of the exact task/machine/SOP: \\\"Tailor needed to stitch kurta sets on the industrial machine, take measurements from customers, and finish button holes by hand.\\\"\\n\\nGenerate the capsule as JSON.\", \"response\": {\"role\": \"Tailor - Kurta Stitching\", \"requiredTrade\": \"Tailoring (Machine + Hand finishing)\", \"skillsNeeded\": [\"Industrial sewing machine\", \"Measurement taking\", \"Hand button-holing\", \"Kurta pattern cutting\"], \"steps\": [{\"title\": \"Take customer measurements\", \"instruction\": \"Record chest, shoulder, sleeve, and length measurements accurately on the order slip.\"}, {\"title\": \"Cut the kurta pieces\", \"instruction\": \"Cut fabric pieces from the pattern, matching the recorded measurements with seam allowance.\"}, {\"title\": \"Stitch on the industrial machine\", \"instruction\": \"Join the kurta panels on the industrial machine with straight, even seams.\"}, {\"title\": \"Finish button holes by hand\", \"instruction\": \"Hand-stitch clean button holes at the collar and front placket.\"}], \"quizQuestions\": [{\"question\": \"Which measurements must be recorded before cutting?\", \"answer\": \"Chest, shoulder, sleeve, and length.\"}, {\"question\": \"How are button holes finished?\", \"answer\": \"By hand-stitching, not machine.\"}]}}",
"{\"system\": \"You are a vocational trainer generating a micro-upskilling capsule for a new hire at an Indian MSME, based on the owner's description of the exact task, machine, or SOP. Return strict JSON only, no markdown. Schema: {\\\"role\\\": short job title, \\\"requiredTrade\\\": closest matching trade name, \\\"skillsNeeded\\\": array of 3-5 short tags, \\\"steps\\\": array of 4-6 objects {\\\"title\\\": short imperative title, \\\"instruction\\\": one to two sentences specific to what the owner described}, \\\"quizQuestions\\\": array of 2-3 objects {\\\"question\\\": string, \\\"answer\\\": string}}. Never write generic trade advice \u2014 every step must trace back to specific details the owner gave.\", \"user\": \"MSME name: Ganga Auto Repairs\\nSector: Automobile\\nLocation: Lucknow, UP\\nWage band: \u20b910,000-14,000/mo\\nOwner's description of the exact task/machine/SOP: \\\"Need a mechanic to do two-wheeler engine oil changes, chain adjustment, and basic carburetor cleaning for customer bikes.\\\"\\n\\nGenerate the capsule as JSON.\", \"response\": {\"role\": \"Two-Wheeler Service Mechanic\", \"requiredTrade\": \"Automobile Mechanic (Two-Wheeler)\", \"skillsNeeded\": [\"Engine oil change\", \"Chain adjustment\", \"Carburetor cleaning\", \"Basic diagnostics\"], \"steps\": [{\"title\": \"Drain and refill engine oil\", \"instruction\": \"Drain old engine oil fully, replace the filter if due, and refill with the correct grade oil.\"}, {\"title\": \"Adjust the drive chain\", \"instruction\": \"Check chain slack and adjust the tensioner to the manufacturer's recommended play.\"}, {\"title\": \"Clean the carburetor\", \"instruction\": \"Remove the carburetor, clean the jets with cleaner spray, and refit without losing gasket seating.\"}, {\"title\": \"Test ride the bike\", \"instruction\": \"Start the bike and test ride briefly to confirm smooth idling and gear shifts.\"}], \"quizQuestions\": [{\"question\": \"What should you check before adjusting the chain?\", \"answer\": \"The current chain slack against the recommended play.\"}, {\"question\": \"What is done after cleaning the carburetor jets?\", \"answer\": \"Refit the carburetor carefully without losing gasket seating.\"}]}}",
"{\"system\": \"You are a vocational trainer generating a micro-upskilling capsule for a new hire at an Indian MSME, based on the owner's description of the exact task, machine, or SOP. Return strict JSON only, no markdown. Schema: {\\\"role\\\": short job title, \\\"requiredTrade\\\": closest matching trade name, \\\"skillsNeeded\\\": array of 3-5 short tags, \\\"steps\\\": array of 4-6 objects {\\\"title\\\": short imperative title, \\\"instruction\\\": one to two sentences specific to what the owner described}, \\\"quizQuestions\\\": array of 2-3 objects {\\\"question\\\": string, \\\"answer\\\": string}}. Never write generic trade advice \u2014 every step must trace back to specific details the owner gave.\", \"user\": \"MSME name: Priya Electricals\\nSector: Electrical\\nLocation: Pune, MH\\nWage band: \u20b913,000-16,000/mo\\nOwner's description of the exact task/machine/SOP: \\\"Electrician helper to wire new residential flats \u2014 laying conduit, pulling wire, and fixing switchboards to the load chart provided by the site engineer.\\\"\\n\\nGenerate the capsule as JSON.\", \"response\": {\"role\": \"Electrician Helper - Residential Wiring\", \"requiredTrade\": \"Electrician (Residential)\", \"skillsNeeded\": [\"Conduit laying\", \"Wire pulling\", \"Switchboard fixing\", \"Load chart reading\"], \"steps\": [{\"title\": \"Read the load chart\", \"instruction\": \"Check the site engineer's load chart to confirm wire gauge and circuit count before starting.\"}, {\"title\": \"Lay the conduit\", \"instruction\": \"Fix conduit pipes along the marked wall/ceiling path per the layout.\"}, {\"title\": \"Pull the wiring\", \"instruction\": \"Pull the correct gauge wire through the conduit without nicking the insulation.\"}, {\"title\": \"Fix the switchboard\", \"instruction\": \"Mount and wire the switchboard exactly matching the circuit labels on the load chart.\"}], \"quizQuestions\": [{\"question\": \"What must be checked before choosing wire gauge?\", \"answer\": \"The site engineer's load chart.\"}, {\"question\": \"What must be avoided while pulling wire through conduit?\", \"answer\": \"Nicking or damaging the wire insulation.\"}]}}",
"{\"system\": \"You are an experienced trade examiner for Indian MSME skilled trades. You evaluate a worker's spoken description of how they perform a task and produce a strict JSON object only, no markdown, no preamble. Schema: {\\\"score\\\": integer 0-900, \\\"level\\\": one of \\\"Novice\\\",\\\"Competent\\\",\\\"Skilled\\\",\\\"Master\\\", \\\"strengths\\\": array of 2-3 short strings, \\\"gaps\\\": array of 1-2 short strings, \\\"followUpQuestion\\\": a short adaptive oral viva follow-up question probing the weakest point, \\\"credentialSummary\\\": one plain sentence suitable for a verifiable credential}. Be realistic: vague or generic answers score under 400; specific, technically correct, safety-aware answers score 700-900.\", \"user\": \"Trade: Welder\\nLanguage spoken: Tamil\\nWorker's spoken description (transcribed): \\\"Naan MS plate weld panren, current set pandren electrode size paathu, shield potutu weld panren, edge clean panren mudinjadhukku apram.\\\"\\n\\nEvaluate and return only the JSON.\", \"response\": {\"score\": 640, \"level\": \"Skilled\", \"strengths\": [\"Correct electrode-current matching\", \"Uses safety shield consistently\"], \"gaps\": [\"Did not mention checking penetration or slag removal quality\"], \"followUpQuestion\": \"How do you check if the weld penetration is deep enough on a thick MS plate?\", \"credentialSummary\": \"Demonstrates correct arc-welding safety practice and electrode selection for MS plate work.\"}}",
"{\"system\": \"You are an experienced trade examiner for Indian MSME skilled trades. You evaluate a worker's spoken description of how they perform a task and produce a strict JSON object only, no markdown, no preamble. Schema: {\\\"score\\\": integer 0-900, \\\"level\\\": one of \\\"Novice\\\",\\\"Competent\\\",\\\"Skilled\\\",\\\"Master\\\", \\\"strengths\\\": array of 2-3 short strings, \\\"gaps\\\": array of 1-2 short strings, \\\"followUpQuestion\\\": a short adaptive oral viva follow-up question probing the weakest point, \\\"credentialSummary\\\": one plain sentence suitable for a verifiable credential}. Be realistic: vague or generic answers score under 400; specific, technically correct, safety-aware answers score 700-900.\", \"user\": \"Trade: Tailor\\nLanguage spoken: Hindi\\nWorker's spoken description (transcribed): \\\"Main kapda kaat ke silai machine pe seedha seam banata hoon, button hole thoda mushkil lagta hai abhi seekh raha hoon.\\\"\\n\\nEvaluate and return only the JSON.\", \"response\": {\"score\": 380, \"level\": \"Novice\", \"strengths\": [\"Comfortable with straight-seam stitching on the machine\"], \"gaps\": [\"Admits button-hole finishing is still weak\", \"No mention of measurement-taking accuracy\"], \"followUpQuestion\": \"How do you take chest and sleeve measurements accurately before cutting fabric?\", \"credentialSummary\": \"Can stitch straight machine seams but button-hole finishing needs further training.\"}}",
"{\"system\": \"You are an experienced trade examiner for Indian MSME skilled trades. You evaluate a worker's spoken description of how they perform a task and produce a strict JSON object only, no markdown, no preamble. Schema: {\\\"score\\\": integer 0-900, \\\"level\\\": one of \\\"Novice\\\",\\\"Competent\\\",\\\"Skilled\\\",\\\"Master\\\", \\\"strengths\\\": array of 2-3 short strings, \\\"gaps\\\": array of 1-2 short strings, \\\"followUpQuestion\\\": a short adaptive oral viva follow-up question probing the weakest point, \\\"credentialSummary\\\": one plain sentence suitable for a verifiable credential}. Be realistic: vague or generic answers score under 400; specific, technically correct, safety-aware answers score 700-900.\", \"user\": \"Trade: Plumber\\nLanguage spoken: English\\nWorker's spoken description (transcribed): \\\"I fix leaking pipe joints using Teflon tape and pipe wrench, I check water pressure after fixing, and I always shut the main valve before starting any repair.\\\"\\n\\nEvaluate and return only the JSON.\", \"response\": {\"score\": 780, \"level\": \"Skilled\", \"strengths\": [\"Uses correct sealing method (Teflon tape)\", \"Follows safety step of shutting main valve first\"], \"gaps\": [\"Did not mention how leaks are diagnosed before disassembly\"], \"followUpQuestion\": \"How do you identify exactly where a hidden pipe leak is coming from before opening the joint?\", \"credentialSummary\": \"Follows correct safety sequencing and sealing technique for residential pipe joint repair.\"}}",
"{\"system\": \"You are an experienced trade examiner for Indian MSME skilled trades. You evaluate a worker's spoken description of how they perform a task and produce a strict JSON object only, no markdown, no preamble. Schema: {\\\"score\\\": integer 0-900, \\\"level\\\": one of \\\"Novice\\\",\\\"Competent\\\",\\\"Skilled\\\",\\\"Master\\\", \\\"strengths\\\": array of 2-3 short strings, \\\"gaps\\\": array of 1-2 short strings, \\\"followUpQuestion\\\": a short adaptive oral viva follow-up question probing the weakest point, \\\"credentialSummary\\\": one plain sentence suitable for a verifiable credential}. Be realistic: vague or generic answers score under 400; specific, technically correct, safety-aware answers score 700-900.\", \"user\": \"Trade: Electrician\\nLanguage spoken: English\\nWorker's spoken description (transcribed): \\\"I connect wires.\\\"\\n\\nEvaluate and return only the JSON.\", \"response\": {\"score\": 220, \"level\": \"Novice\", \"strengths\": [\"Aware of the basic task\"], \"gaps\": [\"No detail on safety steps, tools used, or how connections are verified\"], \"followUpQuestion\": \"Before connecting any wire, what is the very first safety step you take?\", \"credentialSummary\": \"Answer too generic to confirm practical competence; needs a detailed follow-up assessment.\"}}",
"{\"system\": \"You write concise, professional one-page resumes for skilled tradespeople applying to Indian MSMEs. Write the ENTIRE resume content in English, using natural, correctly spelled vocabulary a native reader would expect (script: Latin). Return strict JSON only, no markdown, no preamble. Schema: {\\\"fullName\\\": string, \\\"headline\\\": short professional title, \\\"summary\\\": 2-3 sentences, \\\"keySkills\\\": array of 4-6 short strings, \\\"experienceHighlights\\\": array of 2-4 short strings framed as accomplishments, \\\"credentialLine\\\": one sentence citing the verified score, \\\"contactNote\\\": one short line, e.g. availability and location}.\", \"user\": \"Worker name: Ravi Kumar\\nTrade: Welder\\nLocation: Coimbatore, TN\\nSkill Credit Score: 780/900 (Skilled)\\nStrengths: Correct electrode-current matching; consistent safety practice; clean MS fabrication finish\\nCredential summary: Demonstrates correct arc-welding safety practice and electrode selection for MS plate work.\\n\\nWrite the resume JSON in English.\", \"response\": {\"fullName\": \"Ravi Kumar\", \"headline\": \"Skilled Arc Welder - MS Fabrication\", \"summary\": \"Skilled welder with hands-on experience in MS angle fabrication and safety-compliant arc welding. Known for precise electrode-current matching and clean finish work.\", \"keySkills\": [\"Arc/MMA welding\", \"MS fabrication\", \"Electrode selection\", \"Workshop safety\"], \"experienceHighlights\": [\"Delivered consistent, safety-compliant welds on gate and grill frames\", \"Verified skill credential score of 780/900\"], \"credentialLine\": \"Verified Skill Credit Score of 780/900 (Skilled) on the SETU.AI trade assessment.\", \"contactNote\": \"Available immediately, based in Coimbatore, Tamil Nadu.\"}}",
"{\"system\": \"You write concise, professional one-page resumes for skilled tradespeople applying to Indian MSMEs. Write the ENTIRE resume content in Hindi, using natural, correctly spelled vocabulary a native reader would expect (script: Devanagari). Return strict JSON only, no markdown, no preamble. Schema: {\\\"fullName\\\": string, \\\"headline\\\": short professional title, \\\"summary\\\": 2-3 sentences, \\\"keySkills\\\": array of 4-6 short strings, \\\"experienceHighlights\\\": array of 2-4 short strings framed as accomplishments, \\\"credentialLine\\\": one sentence citing the verified score, \\\"contactNote\\\": one short line, e.g. availability and location}.\", \"user\": \"Worker name: Suresh Yadav\\nTrade: Plumber\\nLocation: Lucknow, UP\\nSkill Credit Score: 780/900 (Skilled)\\nStrengths: Sahi sealing technique; safety sequence ka palan; leak diagnosis mein anubhav\\nCredential summary: Follows correct safety sequencing and sealing technique for residential pipe joint repair.\\n\\nWrite the resume JSON in Hindi.\", \"response\": {\"fullName\": \"\u0938\u0941\u0930\u0947\u0936 \u092f\u093e\u0926\u0935\", \"headline\": \"\u0915\u0941\u0936\u0932 \u092a\u094d\u0932\u0902\u092c\u0930 - \u0906\u0935\u093e\u0938\u0940\u092f \u092a\u093e\u0907\u092a \u092e\u0930\u092e\u094d\u092e\u0924\", \"summary\": \"\u0905\u0928\u0941\u092d\u0935\u0940 \u092a\u094d\u0932\u0902\u092c\u0930 \u091c\u094b \u0938\u0939\u0940 \u0938\u0940\u0932\u093f\u0902\u0917 \u0924\u0915\u0928\u0940\u0915 \u0914\u0930 \u0938\u0941\u0930\u0915\u094d\u0937\u093e \u092a\u094d\u0930\u0915\u094d\u0930\u093f\u092f\u093e \u0915\u093e \u092a\u093e\u0932\u0928 \u0915\u0930\u0924\u0947 \u0939\u0948\u0902\u0964 \u092a\u093e\u0907\u092a \u091c\u094b\u0921\u093c\u094b\u0902 \u0915\u0940 \u092e\u0930\u092e\u094d\u092e\u0924 \u092e\u0947\u0902 \u0926\u0915\u094d\u0937\u0964\", \"keySkills\": [\"\u092a\u093e\u0907\u092a \u091c\u094b\u0921\u093c \u092e\u0930\u092e\u094d\u092e\u0924\", \"\u091f\u0947\u092b\u094d\u0932\u0949\u0928 \u091f\u0947\u092a \u0938\u0940\u0932\u093f\u0902\u0917\", \"\u091c\u0932 \u0926\u093e\u092c \u091c\u093e\u0902\u091a\", \"\u0938\u0941\u0930\u0915\u094d\u0937\u093e \u092a\u094d\u0930\u0915\u094d\u0930\u093f\u092f\u093e \u092a\u093e\u0932\u0928\"], \"experienceHighlights\": [\"\u0906\u0935\u093e\u0938\u0940\u092f \u092a\u093e\u0907\u092a \u091c\u094b\u0921\u093c\u094b\u0902 \u0915\u0940 \u0935\u093f\u0936\u094d\u0935\u0938\u0928\u0940\u092f \u092e\u0930\u092e\u094d\u092e\u0924\", \"780/900 \u0938\u0924\u094d\u092f\u093e\u092a\u093f\u0924 \u0938\u094d\u0915\u093f\u0932 \u0938\u094d\u0915\u094b\u0930 \u092a\u094d\u0930\u093e\u092a\u094d\u0924\"], \"credentialLine\": \"SETU.AI \u091f\u094d\u0930\u0947\u0921 \u092e\u0942\u0932\u094d\u092f\u093e\u0902\u0915\u0928 \u092e\u0947\u0902 780/900 (\u0915\u0941\u0936\u0932) \u0938\u0924\u094d\u092f\u093e\u092a\u093f\u0924 \u0938\u094d\u0915\u093f\u0932 \u0915\u094d\u0930\u0947\u0921\u093f\u091f \u0938\u094d\u0915\u094b\u0930\u0964\", \"contactNote\": \"\u0924\u0941\u0930\u0902\u0924 \u0909\u092a\u0932\u092c\u094d\u0927, \u0932\u0916\u0928\u090a, \u0909\u0924\u094d\u0924\u0930 \u092a\u094d\u0930\u0926\u0947\u0936 \u092e\u0947\u0902 \u0938\u094d\u0925\u093f\u0924\u0964\"}}",
"{\"system\": \"You write concise, professional one-page resumes for skilled tradespeople applying to Indian MSMEs. Write the ENTIRE resume content in Tamil, using natural, correctly spelled vocabulary a native reader would expect (script: Tamil). Return strict JSON only, no markdown, no preamble. Schema: {\\\"fullName\\\": string, \\\"headline\\\": short professional title, \\\"summary\\\": 2-3 sentences, \\\"keySkills\\\": array of 4-6 short strings, \\\"experienceHighlights\\\": array of 2-4 short strings framed as accomplishments, \\\"credentialLine\\\": one sentence citing the verified score, \\\"contactNote\\\": one short line, e.g. availability and location}.\", \"user\": \"Worker name: Muthu Raman\\nTrade: Welder\\nLocation: Coimbatore, TN\\nSkill Credit Score: 640/900 (Skilled)\\nStrengths: Sariyaana electrode-current matching; pathukappu murai palikkiravar\\nCredential summary: Demonstrates correct arc-welding safety practice and electrode selection for MS plate work.\\n\\nWrite the resume JSON in Tamil.\", \"response\": {\"fullName\": \"\u0bae\u0bc1\u0ba4\u0bcd\u0ba4\u0bc1 \u0bb0\u0bbe\u0bae\u0ba9\u0bcd\", \"headline\": \"\u0ba4\u0bbf\u0bb1\u0bae\u0bc8\u0baf\u0bbe\u0ba9 \u0b86\u0bb0\u0bcd\u0b95\u0bcd \u0bb5\u0bc6\u0bb2\u0bcd\u0b9f\u0bb0\u0bcd - \u0b8e\u0bae\u0bcd\u0b8e\u0bb8\u0bcd \u0baa\u0bc7\u0baa\u0bcd\u0bb0\u0bbf\u0b95\u0bc7\u0bb7\u0ba9\u0bcd\", \"summary\": \"\u0b8e\u0bae\u0bcd\u0b8e\u0bb8\u0bcd \u0b86\u0b99\u0bcd\u0b95\u0bbf\u0bb3\u0bcd \u0baa\u0bc7\u0baa\u0bcd\u0bb0\u0bbf\u0b95\u0bc7\u0bb7\u0ba9\u0bbf\u0bb2\u0bcd \u0b95\u0bc8\u0bb5\u0bb0\u0bbf\u0b9a\u0bc8 \u0baa\u0bc6\u0bb1\u0bcd\u0bb1 \u0bb5\u0bc6\u0bb2\u0bcd\u0b9f\u0bb0\u0bcd. \u0b9a\u0bb0\u0bbf\u0baf\u0bbe\u0ba9 \u0b8e\u0bb2\u0bc6\u0b95\u0bcd\u0b9f\u0bcd\u0bb0\u0bcb\u0b9f\u0bcd-\u0b95\u0bb0\u0ba3\u0bcd\u0b9f\u0bcd \u0baa\u0bca\u0bb0\u0bc1\u0ba4\u0bcd\u0ba4\u0bae\u0bc1\u0bae\u0bcd \u0baa\u0bbe\u0ba4\u0bc1\u0b95\u0bbe\u0baa\u0bcd\u0baa\u0bc1 \u0bae\u0bc1\u0bb1\u0bc8\u0baf\u0bc1\u0bae\u0bcd \u0baa\u0bbf\u0ba9\u0bcd\u0baa\u0bb1\u0bcd\u0bb1\u0bc1\u0baa\u0bb5\u0bb0\u0bcd.\", \"keySkills\": [\"\u0b86\u0bb0\u0bcd\u0b95\u0bcd \u0bb5\u0bc6\u0bb2\u0bcd\u0b9f\u0bbf\u0b99\u0bcd\", \"\u0b8e\u0bae\u0bcd\u0b8e\u0bb8\u0bcd \u0baa\u0bc7\u0baa\u0bcd\u0bb0\u0bbf\u0b95\u0bc7\u0bb7\u0ba9\u0bcd\", \"\u0b8e\u0bb2\u0bc6\u0b95\u0bcd\u0b9f\u0bcd\u0bb0\u0bcb\u0b9f\u0bcd \u0ba4\u0bc7\u0bb0\u0bcd\u0bb5\u0bc1\", \"\u0baa\u0ba3\u0bbf\u0bae\u0ba9\u0bc8 \u0baa\u0bbe\u0ba4\u0bc1\u0b95\u0bbe\u0baa\u0bcd\u0baa\u0bc1\"], \"experienceHighlights\": [\"\u0b95\u0bc7\u0b9f\u0bcd \u0bae\u0bb1\u0bcd\u0bb1\u0bc1\u0bae\u0bcd \u0b95\u0bbf\u0bb0\u0bbf\u0bb2\u0bcd \u0b83\u0baa\u0bcd\u0bb0\u0bc7\u0bae\u0bcd\u0b95\u0bb3\u0bbf\u0bb2\u0bcd \u0b9a\u0bc0\u0bb0\u0bbe\u0ba9 \u0bb5\u0bc6\u0bb2\u0bcd\u0b9f\u0bbf\u0b99\u0bcd \u0baa\u0ba3\u0bbf\", \"780/900 \u0b9a\u0bb0\u0bbf\u0baa\u0bbe\u0bb0\u0bcd\u0b95\u0bcd\u0b95\u0baa\u0bcd\u0baa\u0b9f\u0bcd\u0b9f \u0bb8\u0bcd\u0b95\u0bcb\u0bb0\u0bcd\"], \"credentialLine\": \"SETU.AI \u0bae\u0ba4\u0bbf\u0baa\u0bcd\u0baa\u0bc0\u0b9f\u0bcd\u0b9f\u0bbf\u0bb2\u0bcd 640/900 (\u0ba4\u0bbf\u0bb1\u0bae\u0bc8\u0baf\u0bbe\u0ba9) \u0b9a\u0bb0\u0bbf\u0baa\u0bbe\u0bb0\u0bcd\u0b95\u0bcd\u0b95\u0baa\u0bcd\u0baa\u0b9f\u0bcd\u0b9f \u0bb8\u0bcd\u0b95\u0bbf\u0bb2\u0bcd \u0b95\u0bbf\u0bb0\u0bc6\u0b9f\u0bbf\u0b9f\u0bcd \u0bb8\u0bcd\u0b95\u0bcb\u0bb0\u0bcd.\", \"contactNote\": \"\u0b89\u0b9f\u0ba9\u0b9f\u0bbf\u0baf\u0bbe\u0b95 \u0b95\u0bbf\u0b9f\u0bc8\u0b95\u0bcd\u0b95\u0bc1\u0bae\u0bcd, \u0b95\u0bcb\u0baf\u0bae\u0bcd\u0baa\u0bc1\u0ba4\u0bcd\u0ba4\u0bc2\u0bb0\u0bbf\u0bb2\u0bcd \u0b89\u0bb3\u0bcd\u0bb3\u0bbe\u0bb0\u0bcd.\"}}",
"{\"system\": \"You are a matching engine for an Indian labour marketplace. Given a job requirement and a list of candidate workers with trade, skill score (0-900), location, and language, rank the best-fit candidates. Return strict JSON only: {\\\"matches\\\": array of up to 3 objects {\\\"workerId\\\": string, \\\"matchScore\\\": integer 0-100, \\\"rationale\\\": one short sentence citing score, trade fit, and location or language fit}}. Prefer higher skill score, matching trade, and matching or nearby location.\", \"user\": \"Job requirement:\\nRole: Arc Welder Assistant\\nSector: Fabrication\\nLocation: Coimbatore, TN\\nSkills needed: Arc welding, MS fabrication, Safety compliance\\n\\nCandidates:\\nid:w1 | name:Muthu Raman | trade:Welder | score:780 | location:Coimbatore, TN | language:Tamil\\nid:w2 | name:Deepak Singh | trade:Electrician | score:820 | location:Delhi, DL | language:Hindi\\nid:w3 | name:Selvam K | trade:Welder | score:540 | location:Tirupur, TN | language:Tamil\\n\\nReturn ranked matches as JSON.\", \"response\": {\"matches\": [{\"workerId\": \"w1\", \"matchScore\": 92, \"rationale\": \"Top skill score (780), exact trade match, same city as the job.\"}, {\"workerId\": \"w3\", \"matchScore\": 68, \"rationale\": \"Correct trade match and nearby location, but lower skill score (540).\"}]}}",
"{\"system\": \"You are a matching engine for an Indian labour marketplace. Given a job requirement and a list of candidate workers with trade, skill score (0-900), location, and language, rank the best-fit candidates. Return strict JSON only: {\\\"matches\\\": array of up to 3 objects {\\\"workerId\\\": string, \\\"matchScore\\\": integer 0-100, \\\"rationale\\\": one short sentence citing score, trade fit, and location or language fit}}. Prefer higher skill score, matching trade, and matching or nearby location.\", \"user\": \"Job requirement:\\nRole: Two-Wheeler Service Mechanic\\nSector: Automobile\\nLocation: Lucknow, UP\\nSkills needed: Engine oil change, Carburetor cleaning\\n\\nCandidates:\\nid:w4 | name:Suresh Yadav | trade:Plumber | score:780 | location:Lucknow, UP | language:Hindi\\nid:w5 | name:Anil Verma | trade:Automobile Mechanic | score:710 | location:Kanpur, UP | language:Hindi\\nid:w6 | name:Rajesh Tiwari | trade:Automobile Mechanic | score:690 | location:Lucknow, UP | language:Hindi\\n\\nReturn ranked matches as JSON.\", \"response\": {\"matches\": [{\"workerId\": \"w6\", \"matchScore\": 90, \"rationale\": \"Matching automobile trade, same city, strong skill score (690).\"}, {\"workerId\": \"w5\", \"matchScore\": 78, \"rationale\": \"Matching trade with higher score (710) but from a nearby city, not the same one.\"}]}}"
]


## 4. (Optional) Upload real logged usage

Skip this cell entirely for a first run — it's fine to train on seed data
only. Once you have `backend/data/training_log.jsonl` from real SETU.AI
usage, run this cell and upload it to blend it in; more real examples here
means a meaningfully better model.

In [ ]:
import json
from pathlib import Path
from google.colab import files

logged_lines = []
try:
    uploaded = files.upload()  # pick training_log.jsonl, or click Cancel to skip
    for name in uploaded:
        text = uploaded[name].decode("utf-8")
        logged_lines.extend(l for l in text.splitlines() if l.strip())
    print(f"Loaded {len(logged_lines)} real logged examples.")
except Exception as e:
    print("Skipped upload (that's fine):", e)


## 5. Build the training-ready dataset (dedupe + chat format)

In [ ]:
import json

def load_lines(lines):
    out = []
    for line in lines:
        try:
            out.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return out

seed_records = load_lines(SEED_JSONL_LINES)
logged_records = load_lines(logged_lines) if "logged_lines" in globals() else []
print(f"Seed examples: {len(seed_records)}")
print(f"Logged examples: {len(logged_records)}")

combined = seed_records + logged_records  # logged (later) wins on exact-prompt collisions
seen = {}
for rec in combined:
    key = (rec.get("system", "").strip(), rec.get("user", "").strip())
    seen[key] = rec

def to_chat_example(rec):
    return {"messages": [
        {"role": "system", "content": rec["system"]},
        {"role": "user", "content": rec["user"]},
        {"role": "assistant", "content": json.dumps(rec["response"], ensure_ascii=False)},
    ]}

train_examples = [to_chat_example(r) for r in seen.values()]
print(f"Final training examples: {len(train_examples)}")

with open("train.jsonl", "w", encoding="utf-8") as f:
    for ex in train_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

if len(train_examples) < 100:
    print("\nNote: this is enough to prove the pipeline works end to end, but for a model "
          "you'd trust in production, aim for 300-1000+ examples — mostly by turning on "
          "LOG_TRAINING_DATA=true in your backend and re-running this notebook later with "
          "the accumulated real usage log.")


## 6. Load the base model (4-bit, low VRAM)

Default is Qwen2.5-3B-Instruct — good English + Hindi + workable Tamil
coverage, fits a free-tier T4's 16GB VRAM in 4-bit with room to spare. If you
hit an out-of-memory error, switch `BASE_MODEL` to the 1.5B variant
(commented out below) and re-run from here.

In [ ]:
from unsloth import FastLanguageModel

BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
# BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"  # use this if you OOM on a smaller GPU

MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)


## 7. Fine-tune

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

dataset = load_dataset("json", data_files="train.jsonl", split="train")

def format_example(ex):
    text = tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_example)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        logging_steps=5,
        optim="adamw_8bit",
        output_dir="output/setu-ai-lora",
        save_strategy="epoch",
        report_to="none",
    ),
)

trainer.train()


## 8. Save the adapter and merge into full weights

In [ ]:
model.save_pretrained("output/setu-ai-lora")
tokenizer.save_pretrained("output/setu-ai-lora")

model.save_pretrained_merged("output/setu-ai-merged", tokenizer, save_method="merged_16bit")
print("Merged model saved to output/setu-ai-merged")


## 9. Convert to GGUF and quantize

This is the format Ollama loads. Quantizing to Q4_K_M keeps the file small
and fast at inference time on your own machine.

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF
!cmake --build llama.cpp/build --target llama-quantize -j 4

!python llama.cpp/convert_hf_to_gguf.py output/setu-ai-merged --outfile setu-ai-f16.gguf --outtype f16
!llama.cpp/build/bin/llama-quantize setu-ai-f16.gguf setu-ai-Q4_K_M.gguf Q4_K_M

print("Done: setu-ai-Q4_K_M.gguf")


## 10. Download the model

Downloads straight to your computer. For a 3B model this file is roughly
2GB — the download itself may take a few minutes depending on your
connection. (If it's more convenient, you can instead save it to Google
Drive with `!cp setu-ai-Q4_K_M.gguf /content/drive/MyDrive/` after mounting
Drive.)

In [ ]:
from google.colab import files
files.download("setu-ai-Q4_K_M.gguf")


## 11. Load it into Ollama on your own machine

Once `setu-ai-Q4_K_M.gguf` has downloaded, on your local machine:

```bash
# Install Ollama if you haven't already:
curl -fsSL https://ollama.com/install.sh | sh

# In the folder where you saved the downloaded .gguf file, create a Modelfile:
cat > Modelfile << 'EOF'
FROM ./setu-ai-Q4_K_M.gguf
PARAMETER temperature 0.2
PARAMETER num_ctx 2048
PARAMETER stop "<|im_end|>"
TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>
"""
SYSTEM """You are the SETU.AI engine: a model fine-tuned specifically to write vocational training capsules, grade skill assessments, write multilingual resumes, and rank worker-job matches for an Indian MSME labour marketplace. Always reply with a single strict JSON object matching the schema given in the request — no markdown, no preamble, no explanation."""
EOF

ollama create setu-ai -f Modelfile
ollama run setu-ai   # sanity check
```

Then in `backend/.env`:
```
AI_PROVIDER=local
LOCAL_AI_URL=http://localhost:11434
LOCAL_AI_MODEL=setu-ai
```

Restart the backend (`npm start`). SETU.AI now runs on the model you just
trained — no cloud dependency, no per-call cost.